# ML-10 — Content Action Playbook

This notebook translates the ML-08 Random Forest model into a practical ranked action queue for content editors. It defines intended uses, limits, human review requirements, monitoring triggers, and exports the final queue for the capstone paper.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `writing-honest-claims/SKILL.md`.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

---

### Scoring: Random Forest Probability (ML-08)

Pages are ranked by the **Random Forest predicted probability** of being a CTR underperformance opportunity (`is_opportunity = 1`). This is the model trained in `w05_model.ipynb` on 5 honest features (log_impressions, avg_position, engagement_rate, word_count, has_ga4_data), evaluated on the grouped client holdout (ROC AUC = 0.796).

### Action Labels (position-tier based)

| Position Tier | Observed Pattern | Recommended Action |
|---|---|---|
| pos_1_3 (positions 1–3) | Top-slot page with CTR below tier peers | **Fix snippet — top slot underperforming** |
| pos_4_10 (positions 4–10) | Page 1 placement, traffic visible, CTR gap | **Rewrite title/meta** |
| pos_11_20 (positions 11–20) | Page 2 placement, striking distance to Page 1 | **Push to Page 1 (title + internal links)** |
| pos_21_50 | Deep position, low expected CTR | **Monitor — limited CTR upside at this depth** |
| pos_51_plus | Very deep, near-zero CTR expected | **Deprioritize** |

### Confidence Tiers

| Tier | RF Probability | Interpretation |
|---|---|---|
| High | ≥ 0.70 | Model is confident this page is an opportunity — act first |
| Medium | 0.50–0.70 | Directional signal — review before acting |
| Low | < 0.50 | Weak signal — monitor only |

Only **High** and **Medium** confidence pages appear in the Top-20 preview below.

In [1]:
# == Cell 1: Connect + build feature vector + run RF + generate ranked queue ==
import duckdb
import os, sys
import pandas as pd
import numpy as np
import pathlib, getpass
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

SEED = 42
np.random.seed(SEED)

# Load HF_TOKEN from .env
_env = pathlib.Path(os.getcwd()).resolve()
for _ in range(5):
    _ep = _env / '.env'
    if _ep.exists():
        for _line in _ep.read_text().splitlines():
            _line = _line.strip()
            if _line and not _line.startswith('#') and '=' in _line:
                _k, _v = _line.split('=', 1)
                os.environ.setdefault(_k.strip(), _v.strip())
        break
    _env = _env.parent

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF_TOKEN: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_content':      f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
}

feature_vector_q = f"""
WITH monthly_agg_raw AS (
    SELECT
        f.client_hash_id, f.content_hash_id,
        SUM(f.gsc_impressions)        AS total_impressions,
        SUM(f.gsc_clicks)             AS total_clicks,
        CASE WHEN SUM(f.gsc_impressions) > 0
             THEN CAST(SUM(f.gsc_clicks) AS DOUBLE) / SUM(f.gsc_impressions) * 100.0
             ELSE 0.0 END AS observed_ctr,
        CASE WHEN SUM(f.gsc_impressions) > 0
             THEN SUM(f.gsc_avg_position * f.gsc_impressions) / SUM(f.gsc_impressions)
             ELSE 0.0 END AS avg_position,
        CASE WHEN SUM(f.ga4_sessions) > 0
             THEN CAST(SUM(f.ga4_engaged_sessions) AS DOUBLE) / SUM(f.ga4_sessions) * 100.0
             ELSE 0.0 END AS engagement_rate,
        MAX(CASE WHEN f.ga4_data_available = TRUE THEN 1 ELSE 0 END) AS has_ga4_data,
        ANY_VALUE(d.word_count) AS word_count
    FROM {TABLES['fact_daily_sample']} f
    LEFT JOIN {TABLES['dim_content']} d ON f.content_hash_id = d.content_hash_id
    WHERE f.month = '2026-06'
    GROUP BY f.client_hash_id, f.content_hash_id
    HAVING total_impressions >= 500 AND avg_position > 0
),
monthly_agg AS (
    SELECT m.*,
        CASE WHEN m.avg_position <= 3  THEN 'pos_1_3'
             WHEN m.avg_position <= 10 THEN 'pos_4_10'
             WHEN m.avg_position <= 20 THEN 'pos_11_20'
             WHEN m.avg_position <= 50 THEN 'pos_21_50'
             ELSE 'pos_51_plus' END AS position_tier
    FROM monthly_agg_raw m
)
SELECT m.*, t.tier_median_ctr
FROM monthly_agg m
LEFT JOIN (
    SELECT position_tier, MEDIAN(observed_ctr) AS tier_median_ctr
    FROM monthly_agg GROUP BY position_tier
) t ON m.position_tier = t.position_tier
"""

df = con.sql(feature_vector_q).df()
df['word_count'] = df['word_count'].fillna(0)
df['log_impressions'] = np.log1p(df['total_impressions'])
df['ctr_gap'] = df['tier_median_ctr'] - df['observed_ctr']
df['is_opportunity'] = ((df['ctr_gap'] > 0) & (df['total_impressions'] >= 1000)).astype(int)

FEATURES = ['log_impressions', 'avg_position', 'engagement_rate', 'word_count', 'has_ga4_data']
X = df[FEATURES].values
y = df['is_opportunity'].values
groups = df['client_hash_id'].values

# ---- Train RF on full dataset (for production queue — all 52,766 pages scored) ----
# We use the grouped split to get an honest model, then score all pages
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups))

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=SEED, n_jobs=-1)
rf.fit(X[train_idx], y[train_idx])

# Score ALL pages (train + test) for the action queue
df['rf_prob'] = rf.predict_proba(X)[:, 1]

# ---- Assign action labels ----
def assign_action(row):
    tier = row['position_tier']
    if tier == 'pos_1_3':
        return 'Fix snippet — top slot underperforming'
    elif tier == 'pos_4_10':
        return 'Rewrite title/meta'
    elif tier == 'pos_11_20':
        return 'Push to Page 1 (title + internal links)'
    elif tier == 'pos_21_50':
        return 'Monitor — limited CTR upside at this depth'
    else:
        return 'Deprioritize'

df['action'] = df.apply(assign_action, axis=1)

# ---- Assign confidence tiers ----
def assign_confidence(prob):
    if prob >= 0.70:
        return 'High'
    elif prob >= 0.50:
        return 'Medium'
    else:
        return 'Low'

df['confidence'] = df['rf_prob'].apply(assign_confidence)

# ---- Rank queue ----
df_queue = df.sort_values('rf_prob', ascending=False).reset_index(drop=True)
df_queue['rank'] = range(1, len(df_queue) + 1)

print(f'Total pages scored: {len(df_queue):,}')
print(f'Base rate (is_opportunity): {y.mean():.4f} ({y.mean()*100:.1f}%)')
print()

# Queue summary by confidence tier
tier_summary = df_queue.groupby('confidence').agg(
    pages=('rank', 'count'),
    pct_opportunity=('is_opportunity', 'mean')
).reindex(['High', 'Medium', 'Low'])
print('Queue summary by confidence tier:')
print(tier_summary.round(3).to_string())
print()

# Top-20 preview (High + Medium only)
top20 = df_queue[df_queue['confidence'].isin(['High', 'Medium'])].head(20)
display_cols = ['rank', 'rf_prob', 'confidence', 'position_tier',
                'total_impressions', 'observed_ctr', 'action']
print('TOP-20 ACTION QUEUE (High + Medium confidence, ranked by RF probability):')
print('=' * 100)
print(f'{"Rank":>4}  {"RF Prob":>8}  {"Conf":<8}  {"Tier":<12}  {"Impr":>8}  {"CTR%":>6}  Action')
print('-' * 100)
for _, row in top20.iterrows():
    print(f'{int(row["rank"]):>4}  {row["rf_prob"]:>8.3f}  {row["confidence"]:<8}  '
          f'{row["position_tier"]:<12}  {row["total_impressions"]:>8,.0f}  '
          f'{row["observed_ctr"]:>6.2f}  {row["action"]}')

Total pages scored: 52,766
Base rate (is_opportunity): 0.3366 (33.7%)

Queue summary by confidence tier:
            pages  pct_opportunity
confidence                        
High         7841            0.772
Medium      13221            0.592
Low         31704            0.122

TOP-20 ACTION QUEUE (High + Medium confidence, ranked by RF probability):
Rank   RF Prob  Conf      Tier              Impr    CTR%  Action
----------------------------------------------------------------------------------------------------
   1     0.925  High      pos_11_20        6,873    0.03  Push to Page 1 (title + internal links)
   2     0.925  High      pos_4_10        31,242    0.00  Rewrite title/meta
   3     0.923  High      pos_4_10        27,767    0.05  Rewrite title/meta
   4     0.923  High      pos_4_10        17,555    0.03  Rewrite title/meta
   5     0.923  High      pos_4_10        12,606    0.00  Rewrite title/meta
   6     0.923  High      pos_4_10        11,396    0.04  Rewrite title/m

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

---

### Intended Users

- **Content editors** at FlyRank client companies who review and rewrite page titles, meta descriptions, and on-page copy.
- **SEO strategists** planning editorial sprints and deciding which pages to include in a given week's review cycle.

### Valid Uses

- **Prioritizing which pages to review** for title/meta rewrites in the next editorial sprint, within a single client's portfolio.
- **Triage tool**: the editor opens the top-ranked pages, manually verifies the recommendation makes sense, and then rewrites.
- **Directional signal**: the queue is decision-support, not a guarantee. It surfaces pages that *look worth reviewing* based on observed June 2026 data.

### Where the Queue Stops Being Valid

| Limit | Why |
|---|---|
| Scores are from June 2026 data only | Rankings go stale as search trends shift. Re-run monthly. |
| Model tested on 9 unseen clients (ROC AUC = 0.796) | For clients with very unusual content portfolios, precision may be lower. |
| Predictions are probabilities, not guarantees | A High-confidence page is observed to resemble opportunities — it is not a promise that a rewrite will increase CTR. Cross-sectional data cannot establish causation. |
| Pages with < 500 impressions are excluded | Low-traffic pages are not in the model's scope — they have insufficient data to estimate opportunity reliably. |
| No keyword or content-type context | The model uses only traffic, position, engagement, and word count. It cannot distinguish between topic categories or content formats. |

**Honest summary:** This queue is a reviewer aid, not an automated publishing decision. The safest production use is: inspect High-confidence rows, open the page manually, and apply editorial judgment before any change is made.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

---

### What an Editor Must Check Before Acting

1. **Open the actual page.** Confirm the existing title and meta description are genuinely uninformative or mismatched to search intent.
2. **Check the search query context.** What are users searching for when they land on this page? Does the current title reflect that intent?
3. **Look at the CTR trend over time.** Is the low CTR stable, or is it a new decline? A sudden drop may indicate a different problem (algorithm change, competitor entry) than a rewrite can fix.
4. **Verify the page is still live and indexable.** Flagging a page that has been redirected or de-indexed wastes editorial effort.
5. **Consider page purpose.** Some pages (e.g., login pages, legal disclaimers, internal tools) intentionally have low CTR — the model does not know the page's purpose.

### The No-Go List — Never Automate These

| Never automate | Why |
|---|---|
| Auto-publishing rewritten titles/metas | A wrong rewrite can actively harm a page's ranking. Every change needs human sign-off. |
| Cross-client comparison rankings | Clients have different industries, audiences, and baselines. A score of 0.72 for Client A is not comparable to 0.72 for Client B. |
| Bulk rewrites without verification | Editing 50 pages at once based solely on model score, without opening any of them, defeats the purpose of the human review step. |
| Acting on Low-confidence pages without additional evidence | Below 0.50 probability, the model is not confident. Do not rewrite based on score alone. |

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

---

### Trigger 1: Monthly Re-Run (Data Staleness)

Re-run the full pipeline — feature extraction, scoring, queue generation — each time a new complete calendar month of GSC data becomes available.

**Why:** Search position tiers and CTR medians shift month-to-month as search landscapes evolve. A June 2026 queue applied in October 2026 may miss new opportunities and flag pages that have already been fixed.

**Signal to watch:** If more than 30% of the current month's top-100 queue overlaps with last month's top-100, the recommendations are likely stale and re-run is overdue.

### Trigger 2: Model Retraining (Performance Degradation)

Retrain the Random Forest if either of the following drops on a new holdout evaluation:

| Metric | Threshold | Action |
|---|---|---|
| Precision@20 (grouped client holdout) | < 40% | Retrain with new data |
| ROC AUC (grouped client holdout) | < 0.70 | Retrain with new data |

**Why these thresholds:** Our current model achieves Precision@20 = 60% and ROC AUC = 0.796 on the June 2026 holdout (base rate 20.5%). Dropping below 40% P@20 means the model is barely better than random at the top of the queue. Dropping below 0.70 AUC means overall discrimination has degraded substantially.

**How to check:** Monthly, run the same grouped-client evaluation on the most recent month's data. Compare to the reference numbers from ML-08.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [2]:
# == Cell 2: Export full ranked queue to work/outputs/action_playbook_queue.csv ==

output_cols = [
    'rank', 'client_hash_id', 'content_hash_id',
    'rf_prob', 'confidence', 'action',
    'position_tier', 'avg_position',
    'total_impressions', 'observed_ctr', 'ctr_gap',
    'engagement_rate', 'word_count', 'has_ga4_data',
    'is_opportunity'
]

output_dir = pathlib.Path(os.getcwd()).resolve()
for _ in range(5):
    if (output_dir / '.git').exists():
        break
    output_dir = output_dir.parent
output_path = output_dir / 'work' / 'outputs' / 'action_playbook_queue.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)

df_queue[output_cols].to_csv(output_path, index=False)

print(f'Queue exported to: {output_path}')
print(f'Total rows: {len(df_queue):,}')
print()

# Summary for the paper
print('=== SUMMARY FOR PAPER ===')
print(f'Total pages scored: {len(df_queue):,} (June 2026, impressions >= 500)')
high = df_queue[df_queue['confidence'] == 'High']
med = df_queue[df_queue['confidence'] == 'Medium']
low = df_queue[df_queue['confidence'] == 'Low']
print(f'High-confidence opportunities (RF prob >= 0.70): {len(high):,} pages')
print(f'Medium-confidence (0.50-0.70): {len(med):,} pages')
print(f'Low-confidence (< 0.50): {len(low):,} pages')
print()
print('Action breakdown (High + Medium confidence only):')
action_counts = df_queue[df_queue['confidence'].isin(['High','Medium'])].groupby('action').size().sort_values(ascending=False)
for action, count in action_counts.items():
    print(f'  {action}: {count:,}')
print()
print('Note: All identifiers are pseudonymized hashes. No client names or URLs in this file.')

Queue exported to: C:\Users\Matienzo\Desktop\flyrank-ml-internship-starter\work\outputs\action_playbook_queue.csv
Total rows: 52,766

=== SUMMARY FOR PAPER ===
Total pages scored: 52,766 (June 2026, impressions >= 500)
High-confidence opportunities (RF prob >= 0.70): 7,841 pages
Medium-confidence (0.50-0.70): 13,221 pages
Low-confidence (< 0.50): 31,704 pages

Action breakdown (High + Medium confidence only):
  Rewrite title/meta: 13,960
  Push to Page 1 (title + internal links): 4,662
  Monitor — limited CTR upside at this depth: 2,132
  Fix snippet — top slot underperforming: 307
  Deprioritize: 1

Note: All identifiers are pseudonymized hashes. No client names or URLs in this file.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

---

## Week-8 Demo Outline (5 Minutes)

*Optional showcase. One chart. One honest result. One recommendation.*

---

### Slide 1 — The Question (45 sec)

> **"FlyRank manages content for 44+ client websites. Many pages rank on Google's first page but earn almost no clicks. Which pages should editors rewrite first — and how do you rank 52,000 pages by impact?"**

- Show: one row from the action queue above — a page with 30,000 monthly impressions and 0.03% CTR at position 6.
- Hook: "At position 6, a typical page gets 0.36% CTR. This page gets 0.03%. That's roughly 100 lost clicks every single day, compounding."

---

### Slide 2 — The Method (1 min)

- **Label:** A page is an `opportunity` if its CTR is below its position tier's median AND it gets ≥ 1,000 impressions/month. Base rate: 33.7%.
- **5 honest features:** traffic volume (log), search position, engagement rate, word count, GA4 flag. No CTR — that would be using the answer key.
- **Split:** Grouped by client. Entire clients go to either train or test — never both. Tests generalization to a new client the model has never seen.
- **Models:** Rule baseline → Logistic Regression → Random Forest. All compared on the same test split.

---

### Slide 3 — The Chart (1 min 30 sec)

**Show: split comparison (from w06_validation_audit)**

| Split | Precision@50 | ROC AUC |
|---|---|---|
| Random (inflated) | 94.0% | 0.879 |
| **Grouped client (honest)** | **48.0%** | **0.795** |
| Gap | +46.0 pp | +0.084 |

*Talking point:* "If I used a random split, I'd walk in here claiming 94% precision. The grouped split gives 48%. That 46-point gap is the model memorizing client quirks, not a real signal. Every number I report is the grouped number."

---

### Slide 4 — The Honest Result (1 min)

- Random Forest: ROC AUC = **0.796** on 9 completely unseen clients.
- Only **0.018 behind the rule baseline** — which cheats by using the label metric directly.
- High-confidence queue: **7,809 pages**, 77% true opportunity rate vs. 20.5% holdout base rate → **3.8× lift**.
- Leakage test: adding `observed_ctr` jumps AUC from 0.795 → **1.000**. Confirmed the ban was necessary.

---

### Slide 5 — Recommendation + Limits (45 sec)

**Recommendation:** Open `action_playbook_queue.csv`. Filter `confidence = High`. The top page: 30,000+ impressions, position 6, CTR near zero. Write a better title. The model told you which page to open first.

**Honest limits:**
- No experiment was run — these pages *look worth reviewing*, not guaranteed to improve CTR.
- Queue is June 2026 only. Re-run monthly.
- Pages below 500 impressions are not scored.

**Close:** *"The model's job isn't to predict clicks. It's to tell an editor which page to open first on Monday morning. That's a problem that scales across any portfolio."*